# Brain Tumor Classifier using Support Vector Machine (SVM)

- classifies MRI scans as glioma, meningioma, pituitary, or no tumor  
- use an RBF-kernel SVM for efficiency.
- trains on 256x256 grayscale images from images/Training and images/Testing  
- tracks training and validation accuracy each epoch  
- includes a quick test function to predict any single image and show its confidence

In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# load data into two arrays of data and labels
def load_data(root, classes):
    data = []
    labels = []
    for l in classes:
        folder = os.path.join(root, l)
        for d in tqdm(os.listdir(folder), desc=f"Loading {l}"):
            path = os.path.join(folder, d)
            if d.endswith(".npy"):
                data.append(np.load(path).flatten())
                labels.append(l)
    return np.array(data), np.array(labels)

root = os.path.dirname(os.path.abspath("SVM.ipynb"))
training_folder = os.path.join(root, "dataset", "Training")
testing_folder = os.path.join(root, "dataset", "Testing")
classes = ["glioma", "meningioma", "pituitary", "notumor"]
x_train, y_train = load_data(training_folder, classes)
x_test, y_test = load_data(testing_folder, classes)
print(f"Training data shape: {x_train.shape}, Training labels shape: {y_train.shape}")

Loading notumor: 100%|██████████| 810/810 [00:00<00:00, 8893.19it/s]

Training data shape: (5712, 65536), Training labels shape: (5712,)


In [3]:
PCA_N_COMPONENTS = 100

pca = PCA(n_components=PCA_N_COMPONENTS)
x_train_pca = pca.fit_transform(x_train)
x_test_pca = pca.transform(x_test)

svc_rbf = SVC(kernel='rbf', C=10, gamma='scale')
svc_rbf.fit(x_train_pca, y_train)
y_pred = svc_rbf.predict(x_train_pca)
print("Training accuracy (RBF Kernel):", accuracy_score(y_train, y_pred))
y_pred = svc_rbf.predict(x_test_pca)
print("Testing accuracy (RBF Kernel):", accuracy_score(y_test, y_pred))
print("Classification Report (RBF Kernel):")
print(classification_report(y_test, y_pred, target_names=classes))


Training accuracy (RBF Kernel): 1.0
Testing accuracy (RBF Kernel): 0.9641495041952708
Classification Report (RBF Kernel):
              precision    recall  f1-score   support

      glioma       0.96      0.94      0.95       300
  meningioma       0.96      0.92      0.94       306
   pituitary       0.98      1.00      0.99       405
     notumor       0.96      0.99      0.97       300

    accuracy                           0.96      1311
   macro avg       0.96      0.96      0.96      1311
weighted avg       0.96      0.96      0.96      1311

